# LingoPrism – Fine-tune v7: ảnh COCO/LVIS mới + nhãn giả của model gốc cho phần người chưa vẽ

v6 (ảnh COCO *train2017* mới + nhãn LVIS, Open Images ôn lại cân bằng) tăng mạnh chopsticks, pillow, egg, window
nhưng tụt door, bird, bread, bench: trên ảnh COCO chỉ 80 lớp COCO được vẽ đủ, nhãn LVIS thưa, nên mọi vật không có box
bị dạy là *nền*. Xem `AI/report/yolo/finetune_ft6.md`.

v7 = cùng dataset v6 (giải nén lại từ Drive, không dựng lại) + **nhãn giả**: model gốc yolov8m-oiv7 chạy trên các ảnh
COCO mới và thêm box tự tin (≥ `PSEUDO_CONF`) cho các lớp người chưa vẽ (cửa, cửa sổ, mặt người, quần áo…), không
bao giờ thêm cho 80 lớp COCO (đã vẽ đủ). Train ít epoch hơn (v6 đi ngang từ epoch 5).

**Cách chạy**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, cho phép Google Drive.
3. Giữ tab mở. Bị ngắt thì *Run all* lại: dataset và checkpoint nằm trên Drive (`MyDrive/lingoprism_runs`), notebook làm tiếp.
4. Ô cuối tải về `oiv7m-ft-v7.zip` → gửi lại để chấm trên tập test 2 542 ảnh và quyết định có dùng hay không.

**Thời gian (Colab miễn phí, T4):** dựng dataset lần đầu ~1–1,5 giờ (tải nhãn + ảnh Open Images), sau đó giải nén
từ Drive vài phút; train tối đa `HOURS` giờ; đánh giá + export ~10 phút.

In [ ]:
# 1. GPU, mã nguồn, thư viện
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

if not os.path.exists("/content/LingoPrism"):
    !git clone -q https://github.com/GPTii/LingoPrism.git /content/LingoPrism
else:
    !git -C /content/LingoPrism pull -q
%cd /content/LingoPrism
!pip install -q ultralytics ijson pyyaml onnx onnxslim onnxruntime
import torch

assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Cấu hình – có thể chỉnh
import os

os.chdir("/content/LingoPrism")
from google.colab import drive

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/lingoprism_runs"
os.makedirs(RUNS, exist_ok=True)

BASE = "yolov8m-oiv7.pt"  # model server đang chạy (Ultralytics tự tải)
PER_CLASS = 80  # ảnh Open Images mỗi lớp yếu – bằng REPLAY: phần Open Images cân bằng
REPLAY = 80  # ảnh Open Images mỗi lớp còn lại (chống quên)
PER_LABEL = 250  # ảnh COCO/LVIS mới tối đa cho mỗi lớp yếu
THRESHOLD = 0.55  # lớp yếu: AP50 trên tập test dưới ngưỡng này
EPOCHS = 10  # v6: mAP val đi ngang từ epoch 5
HOURS = 3.5  # giới hạn cứng thời gian train
FREEZE = 10  # backbone YOLOv8 = 10 lớp đầu
LR0 = 0.001  # v5 dùng 0.002 và tụt ngay từ epoch 1
BATCH = 16
PSEUDO_CONF = 0.5  # nhãn giả của model gốc: chỉ lấy box từ độ tin cậy này
NAME = "oiv7m-ft-v7"
DS = f"/content/ft_v6_{PER_CLASS}_{PER_LABEL}"  # cùng dataset v6 (đã lưu trên Drive)
TAR = f"{RUNS}/{os.path.basename(DS)}.tar"

In [ ]:
# 3. Dataset: Open Images train (cân bằng) + ảnh COCO train/LVIS mới, nhãn người vẽ, giữ 601 lớp của model gốc
#    Chạy NỀN: đọc thẳng file nhãn (ít RAM), tải ảnh song song; bị dừng thì chạy lại ô này là làm tiếp.
#    Bấm Stop ô này chỉ dừng việc xem tiến độ, việc dựng vẫn chạy. Xong thì lưu .tar lên Drive cho phiên sau.
import os

assert "DS" in globals(), (
    "Runtime vừa khởi động lại (mất biến): bấm Runtime → Run all – dataset và checkpoint trên Drive vẫn còn"
)
os.chdir("/content/LingoPrism")
import json
import shutil
import subprocess
import time


def ready():  # a finished v6 dataset (Open Images part + COCO/LVIS part)
    try:
        return json.load(open(f"{DS}/summary.json")).get("builder") == 3
    except Exception:
        return False


if not ready() and os.path.exists(TAR):
    print("Giải nén dataset từ Drive...")
    !tar -xf {TAR} -C /content
    if not ready():  # bản lưu bị lỗi giữa chừng hoặc từ bản dựng cũ: bỏ đi, dựng lại
        print("Bản lưu trên Drive chưa hoàn chỉnh hoặc là bản cũ → xoá và dựng lại")
        os.remove(TAR)
        shutil.rmtree(DS, ignore_errors=True)
LOG, PID = "/content/build.log", "/content/build.pid"
alive = (
    lambda: os.path.exists(PID)
    and subprocess.run(["kill", "-0", open(PID).read().strip()], capture_output=True).returncode == 0
)
if not ready() and alive() and "v6" not in open(LOG, errors="ignore").read():
    subprocess.run(["pkill", "-f", "build_oiv7_finetune.py"])  # bản dựng v5 còn chạy nền: dừng
    time.sleep(3)
if not ready():
    if not alive():
        subprocess.run(
            f'nohup sh -c "echo v6; python AI/scripts/build_oiv7_finetune.py --base {BASE} --out {DS} --per-class {PER_CLASS} '
            f"--replay {REPLAY} --threshold {THRESHOLD} && python AI/scripts/build_lvis_finetune.py --base {BASE} --out {DS} "
            f'--per-label {PER_LABEL} --threshold {THRESHOLD}" > {LOG} 2>&1 & echo $! > {PID}',
            shell=True,
        )
    pos = 0
    while True:  # theo dõi tiến độ
        time.sleep(15)
        with open(LOG, errors="ignore") as f:
            f.seek(pos)
            new = f.read()
            pos = f.tell()
        for line in new.replace(chr(13), chr(10)).splitlines():  # progress bars use carriage returns
            if line.strip():
                print(line, flush=True)
        if not alive():
            break
    assert ready(), "Dựng dataset bị lỗi – xem 40 dòng cuối của log ở trên: !tail -40 /content/build.log"
    print("Lưu dataset lên Drive...")
    !tar -cf {TAR} --exclude=_work -C /content {os.path.basename(DS)}
# v7: model gốc gán nhãn giả cho phần người chưa vẽ trên ảnh COCO (vài phút, mỗi lần giải nén lại thì làm lại)
!python AI/scripts/pseudo_label_finetune.py --data {DS} --base {BASE} --conf {PSEUDO_CONF}
!rm -f {DS}/labels/*.cache
s = json.load(open(f"{DS}/summary.json"))
print(f"{s['images']} ảnh – {len(s['weak'])} lớp yếu: {', '.join(s['weak'])}")
print(
    f"{len(s['replay'])} lớp ôn lại; {s['images_open_images']} ảnh Open Images + {s['lvis']['images']} ảnh COCO/LVIS mới"
)
print("Nhãn giả:", {k: v for k, v in json.load(open(f"{DS}/pseudo.json")).items() if k != "per_class"})
assert s["images"] > 100, "Quá ít ảnh – xem log: !tail -40 /content/build.log"

In [ ]:
# 4. Fine-tune (tự train tiếp nếu phiên trước bị ngắt) + đo trước / sau trên ảnh val + export ONNX như server
import os

assert "DS" in globals(), (
    "Runtime vừa khởi động lại (mất biến): bấm Runtime → Run all – dataset và checkpoint trên Drive vẫn còn"
)
os.chdir("/content/LingoPrism")
!python AI/scripts/finetune_oiv7.py --data {DS}/data.yaml --base {BASE} --project {RUNS} --name {NAME} \
    --epochs {EPOCHS} --hours {HOURS} --freeze {FREEZE} --lr0 {LR0} --batch {BATCH}

In [ ]:
# 5. Kết quả trên ảnh val (601 lớp, cùng ảnh cho trước và sau)
import os

assert "DS" in globals(), (
    "Runtime vừa khởi động lại (mất biến): bấm Runtime → Run all – dataset và checkpoint trên Drive vẫn còn"
)
os.chdir("/content/LingoPrism")
from IPython.display import Image, display

run = f"{RUNS}/{NAME}"
r = json.load(open(f"{run}/report/finetune_summary.json"))
for k in ("val_before", "val_after"):
    m = r[k]
    print(
        f"{'Trước' if k == 'val_before' else 'Sau  '}: P {m['precision']:.3f} | R {m['recall']:.3f} | mAP50 {m['map50']:.3f} | mAP50-95 {m['map50_95']:.3f}"
    )
if os.path.exists(f"{run}/results.png"):
    display(Image(f"{run}/results.png", width=900))
print("Kết luận cuối cùng chấm trên tập test 2 542 ảnh của đồ án (máy phát triển, xem ô 6).")

In [ ]:
# 6. Nén & tải về → gửi file zip này lại
import os

assert "DS" in globals(), (
    "Runtime vừa khởi động lại (mất biến): bấm Runtime → Run all – dataset và checkpoint trên Drive vẫn còn"
)
os.chdir("/content/LingoPrism")
import glob
import zipfile

from google.colab import files

out = f"/content/{NAME}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{run}/export/*.onnx") + glob.glob(f"{run}/export/model_card.json"):
        z.write(f, os.path.basename(f))
    for f in ("results.csv", "results.png", "args.yaml"):
        if os.path.exists(f"{run}/{f}"):
            z.write(f"{run}/{f}", f)
    for f in glob.glob(f"{run}/report/*.json"):
        z.write(f, "report/" + os.path.basename(f))
    z.write(f"{DS}/summary.json", "summary.json")
    if os.path.exists(f"{DS}/pseudo.json"):
        z.write(f"{DS}/pseudo.json", "pseudo.json")
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")
files.download(out)
# Trên máy phát triển (giải nén vào AI/runs/oiv7m-ft-v7/):
#   python AI/scripts/quantize_onnx.py AI/runs/oiv7m-ft-v7/yolov8m-oiv7-ft-lingoprism.onnx
#   python AI/scripts/eval_testset.py --model ft7=AI/runs/oiv7m-ft-v7/yolov8m-oiv7-ft-lingoprism-int8.onnx
#   python AI/scripts/compare_finetune.py --name ft7 --run AI/runs/oiv7m-ft-v7